# Basic webscraping example

Since we have ~4000 urls to scrape, we should probably have some type of standard throttle with a retry/backoff for difficult links. Might be good to split the data into thirds and run separately to reduce the amount of time for each of us.

## Load data

In [1]:
import pandas as pd

In [2]:
links = pd.read_csv("../data/external/url_only_data.csv")

In [13]:
list(links["url"])

['https://www.foxnews.com/lifestyle/jack-carrs-eisenhower-d-days-memo-noble-undertaking',
 'https://www.foxnews.com/entertainment/bruce-willis-demi-moore-avoided-doing-one-thing-while-co-parenting-daughter-says',
 'https://www.foxnews.com/politics/blinken-meets-with-qatars-prime-minister.print',
 'https://www.foxnews.com/entertainment/emily-blunt-says-toes-curl-when-people-their-kids-want-act-want-say-dont-do-it',
 'https://www.foxnews.com/media/the-view-co-host-cnn-commentator-ana-navarro-host-night-2-democratic-national-convention',
 'https://www.foxnews.com/politics/striking-boeing-workers-boo-after-democrat-sen-maria-cantwell-criticizes-trump.print',
 'https://www.foxnews.com/politics/white-house-grilled-harris-gun-ownership-mandatory-gun-buybacks.print',
 'https://www.foxnews.com/media/tom-cotton-turns-tables-cnns-dana-bash-guns-brings-up-harris-past-remarks-school-shootings',
 'https://www.foxnews.com/politics/kamala-rides-tsunami-positive-press-skeptics-see-risky-choice',
 'http

## Description Example

In [4]:
import requests
from bs4 import BeautifulSoup

url = "https://www.foxnews.com/sports/juan-soto-sends-yankees-world-series-first-time-15-years" # example website

response = requests.get(url)
if response.status_code != 200:
    raise Exception(f"Failed to load page: Status code {response.status_code}")

# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

title = soup.find("h1", class_="headline speakable")

# title should be the following
# Juan Soto sends the Yankees to the World Series for the first time in years

In [6]:
title.get_text()

'Juan Soto sends the Yankees to the World Series for the first time in 15 years'

## Scraping first 100 urls

In [29]:
# Get first 100 titles and store in an array -> do we need to
# throttle requests? it's not hitting the same link right?
import time
from requests.exceptions import RequestException
from collections import defaultdict
import random

def get_headlines(urls: list[str], verbose = False) -> dict[str, list[str]]:
    headlines = defaultdict(list)
    for url in urls:
        time.sleep(random.randint(1,3))
        print(url) if verbose else ""
        source = "fox" if "fox" in url else "nbc"
        
        try: 
            response = requests.get(url)
            response.raise_for_status()
        except RequestException as e:
            print(f"Error {e}. Trying again.")
            
            try:
                time.sleep(5)
                response = requests.get(url)
                response.raise_for_status()
            except:
                print(f"Error again: {e}. Skipping {url}")
                continue
        
        else:
            soup = BeautifulSoup(response.text, "html.parser")
            title = soup.find("h1", class_ = "headline speakable")
            headlines[source].append(title.get_text() if title else None)
        
    return headlines

In [30]:
urls = list(links["url"])[0:50]
headlines = get_headlines(urls = urls)

### Results

In [41]:
import pprint
pprint.pprint(headlines)

defaultdict(<class 'list'>,
            {slice(0, 5, None): [],
             'fox': ["Jack Carr recalls Gen. Eisenhower's D-Day memo about "
                     "'great and noble undertaking'",
                     'Bruce Willis, Demi Moore avoided doing one thing while '
                     'co-parenting, daughter says',
                     None,
                     'Emily Blunt says her ‘toes curl’ when people tell her '
                     "their kids want to act: 'I want to say, don’t do it!'",
                     "'The View' co-host, CNN commentator Ana Navarro to host "
                     'night 2 of Democratic National Convention',
                     None,
                     None,
                     "Tom Cotton turns tables on CNN's Dana Bash on guns, "
                     "brings up Harris' past remarks on school shootings",
                     'Kamala rides tsunami of positive press, but skeptics see '
                     'a risky choice',
                    